In [1]:
import numpy as np
import pandas as pd 
import os
import re
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from sklearn.preprocessing import minmax_scale
import IPython.display as ipd
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, recall_score

In [2]:
import pandas as pd
dataf = pd.read_csv('/kaggle/input/mfcc-of-slurred-vocal/MFCCoutput.csv')
dataf.head()

,Unnamed: 0,0,1,2,3,4,5,6,7,8,...,119,120,121,122,123,124,125,126,127,class
0,0,-398.97055,92.846680,-6.212745,19.836878,-3.015201,9.478265,5.134083,6.716100,1.437088,...,0.018726,-0.244671,-0.650705,-0.262222,0.137795,-0.539087,-0.257223,-0.230522,-0.188446,1.0
1,1,-232.39255,115.044525,-21.028315,39.190132,-17.016842,7.619745,-2.724971,4.140653,-1.230497,...,0.197640,0.564010,0.166091,-0.022294,0.257573,0.168772,0.264451,-0.257519,-0.390595,1.0
2,2,-466.48450,89.272385,-8.458461,30.776363,-11.168960,18.305796,0.989266,10.417193,0.574027,...,0.232031,0.074036,0.237405,0.154122,0.281900,0.768758,-0.073312,0.096406,0.285212,1.0
3,3,-466.73505,62.805060,12.439709,29.304922,12.614448,9.676723,1.418015,13.074185,0.037665,...,0.408432,-0.018611,-0.274066,0.071348,0.231881,0.456909,0.000649,-0.379244,-0.105873,1.0
4,4,-426.44970,80.985410,-4.792509,36.350883,-0.092283,19.156744,-5.060070,10.123955,2.196594,...,-0.064134,0.120973,0.070115,0.138273,0.084424,0.080933,-0.295089,-0.199056,0.120445,1.0


In [3]:
dataf.loc[dataf['class']=='non_dysarthria','class'] = 0.0
dataf.loc[dataf['class']=='dysarthria','class'] = 1.0
dataf['class'] = dataf['class'].astype(float)

X = dataf.iloc[:,:-1].values
y = dataf.iloc[:,-1]

In [4]:
X.shape, y.shape

((4314, 129), (4314,))

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)


In [6]:
print(X_train.shape)
print(X_test.shape)

(3019, 129)
(1295, 129)


In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier, VotingClassifier

In [8]:
hyperparameters_RFC = {'n_estimators': 600, 'max_depth': 90, 'min_samples_split': 6, 'min_samples_leaf': 3,
                       'max_features': 'sqrt', 'bootstrap': False, 'criterion': 'entropy'}

hyperparameters_XGB = {'max_depth': 9,
                       'min_child_weight': 1,
                       'learning_rate': 0.2,
                       'subsample': 0.8,
                       'colsample_bytree': 1.0,
                       'gamma': 0,
                       'n_estimators': 600,
                       'use_label_encoder': False,
                       'eval_metric': 'rmse',
                       'objective': 'binary:logistic'}

hyperparameters_CB = {'bagging_temperature': 0.8607305832563434, 'bootstrap_type': 'MVS',
                      'colsample_bylevel': 0.917411003148779,
                      'depth': 8, 'grow_policy': 'SymmetricTree', 'iterations': 918, 'l2_leaf_reg': 8,
                      'learning_rate': 0.29287291117375575, 'max_bin': 231, 'min_data_in_leaf': 9, 'od_type': 'Iter',
                      'od_wait': 21, 'one_hot_max_size': 7, 'random_strength': 0.6963042728397884,
                      'scale_pos_weight': 1.924541179848884, 'subsample': 0.6480869299533999}

rf = RandomForestClassifier(**hyperparameters_RFC, random_state=150)
cb = CatBoostClassifier(**hyperparameters_CB)
xgb = XGBClassifier(**hyperparameters_XGB)

In [9]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Assuming you have a feature matrix 'X' and target labels 'y'
X = dataf.iloc[:, :-1]  # All columns except the last one
y = dataf.iloc[:, -1]   # Last column as target

# Define classifiers with hyperparameters
rf = RandomForestClassifier(**hyperparameters_RFC, random_state=150)
cb = CatBoostClassifier(**hyperparameters_CB, verbose=0)  # Suppress CatBoost logs
xgb = XGBClassifier(**hyperparameters_XGB)

# Step 1: Train the classifiers
rf.fit(X, y)
cb.fit(X, y)
xgb.fit(X, y)

# Step 2: Get feature importance scores
rf_importance = rf.feature_importances_
cb_importance = cb.get_feature_importance()
xgb_importance = xgb.feature_importances_

# Step 3: Convert to DataFrame
feature_names = dataf.columns[:-1]  # Assuming last column is 'class'
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'RF_Importance': rf_importance,
    'CB_Importance': cb_importance,
    'XGB_Importance': xgb_importance
})

# Normalize importance scores 
importance_df[['RF_Importance', 'CB_Importance', 'XGB_Importance']] = \
    importance_df[['RF_Importance', 'CB_Importance', 'XGB_Importance']].apply(lambda x: x / np.max(x))

# Compute average importance
importance_df['Avg_Importance'] = importance_df[['RF_Importance', 'CB_Importance', 'XGB_Importance']].mean(axis=1)

# Sort by Avg_Importance
importance_df = importance_df.sort_values(by='Avg_Importance', ascending=False).reset_index(drop=True)

# Step 7: Assign Fibonacci weights
def fibonacci(n):
    fib_series = [1, 1]
    for _ in range(n-2):
        fib_series.append(fib_series[-1] + fib_series[-2])
    return fib_series

fib_weights = fibonacci(len(importance_df))

# Compute final ranking score
importance_df['Fibonacci_Weight'] = fib_weights
importance_df['Final_Rank_Score'] = importance_df['Avg_Importance'] * importance_df['Fibonacci_Weight']

# Sort based on Final Rank Score
importance_df = importance_df.sort_values(by='Final_Rank_Score', ascending=False).reset_index(drop=True)

# top ranked features
print(importance_df[['Feature', 'Final_Rank_Score']].head(20))


   Feature            Final_Rank_Score
0      125  525975167546283979177984.0
1      123  436442195637729790787584.0
2      110  304198617782906439860224.0
3       75  223798768734384032841728.0
4      127  139488736316376062361600.0
5        6   88415351200135707099136.0
6       13   57786512804691396526080.0
7       80   36356337546997503361024.0
8      121   22538396035137035829248.0
9       11   15778915241111622516736.0
10      39   10115229613916799631360.0
11     126    6384798978428978069504.0
12      74    4170040565281181401088.0
13      18    2726887365634454716416.0
14     122    1720059290351751135232.0
15      92    1081023291755640258560.0
16      73     672983516424918138880.0
17      84     447082864784966942720.0
18     105     276896263285784903680.0
19       5     172569985266296455168.0


In [10]:
ranked_features=importance_df

In [11]:
print(importance_df.isna().sum())


Feature             0
RF_Importance       0
CB_Importance       0
XGB_Importance      0
Avg_Importance      0
Fibonacci_Weight    0
Final_Rank_Score    0
dtype: int64
